# 03 — Risk Analysis

Deep-dive into risk metrics:

1. Drawdown analysis
2. Rolling 30-day 99% Value-at-Risk
3. Stress test: simulate a 70% BTC price crash

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.fetch_price import fetch_price_data
from data.fetch_onchain import fetch_hashrate
from models.production_cost import compute_production_cost
from signals.signal_engine import compute_signals
from risk.risk_manager import (
    compute_var,
    is_circuit_breaker_active,
    RiskParameters,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True

## 1. Load data

In [ ]:
price_df = fetch_price_data()
hr_df    = fetch_hashrate()
cost_df  = compute_production_cost(hr_df)
sigs     = compute_signals(price_df, cost_df)

close   = sigs['close']
returns = close.pct_change().dropna()
print(f"Daily returns: {len(returns)} observations")
returns.describe()

## 2. Drawdown Analysis

In [ ]:
rolling_max = close.expanding().max()
drawdown = (close - rolling_max) / rolling_max

fig, ax = plt.subplots()
drawdown.plot(ax=ax, color='red', linewidth=0.8)
ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color='red')
ax.set_title('BTC/USDT Historical Drawdown from All-Time High')
ax.set_ylabel('Drawdown')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
plt.tight_layout()
plt.show()

print(f"\nMax drawdown: {drawdown.min()*100:.1f}%")
print(f"Current drawdown from ATH: {drawdown.iloc[-1]*100:.1f}%")

## 3. Rolling 30-day 99% Value-at-Risk

In [ ]:
PORTFOLIO_VALUE = 10_000.0
params = RiskParameters()

# Compute rolling VaR
rolling_sigma = returns.rolling(window=30, min_periods=15).std(ddof=1)
rolling_var = PORTFOLIO_VALUE * rolling_sigma * params.var_confidence_z

fig, ax = plt.subplots()
rolling_var.plot(ax=ax, color='darkorange', linewidth=1)
ax.set_title('Rolling 30-day 99% VaR (USD, $10,000 portfolio)')
ax.set_ylabel('VaR (USD)')
plt.tight_layout()
plt.show()

latest_var = compute_var(PORTFOLIO_VALUE, returns)
print(f"\nLatest 99% 1-day VaR: ${latest_var:,.0f}  ({latest_var/PORTFOLIO_VALUE*100:.1f}% of portfolio)")

## 4. Stress Test — 70% BTC Price Crash

In [ ]:
# Simulate a 70% crash from current price over 90 days
last_price = close.iloc[-1]
crash_target = last_price * 0.30
n_days = 90
crash_prices = pd.Series(
    np.geomspace(last_price, crash_target, n_days),
    index=pd.date_range(close.index[-1], periods=n_days, freq='D', tz='UTC'),
)

# Compute PCR during crash
last_cost = sigs['production_cost'].iloc[-1]
crash_pcr = crash_prices / last_cost

# VaR during crash (using crash return distribution)
crash_returns = crash_prices.pct_change().dropna()
crash_sigma = float(crash_returns.std(ddof=1))
stress_var = PORTFOLIO_VALUE * crash_sigma * params.var_confidence_z

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(14, 8))
axes[0].plot(crash_prices.index, crash_prices.values, color='red')
axes[0].axhline(last_cost, color='green', linestyle='--', label=f'Production cost ≈ ${last_cost:,.0f}')
axes[0].set_title('Stress Test: 70% BTC Crash Scenario')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

axes[1].plot(crash_pcr.index, crash_pcr.values, color='darkorange')
axes[1].axhline(0.9, color='green', linestyle='--', label='Strong Buy (0.9)')
axes[1].set_ylabel('PCR')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nStress test summary:")
print(f"  Crash price target : ${crash_target:,.0f}")
print(f"  Production cost    : ${last_cost:,.0f}")
print(f"  Min PCR in crash   : {crash_pcr.min():.3f}")
print(f"  Signal at trough   : {'STRONG_BUY' if crash_pcr.min() < 0.9 else 'WEAK_BUY' if crash_pcr.min() < 1.1 else 'HOLD'}")
print(f"  Daily σ (crash)    : {crash_sigma*100:.1f}%")
print(f"  Stress 99% VaR     : ${stress_var:,.0f}")